# Challenge 4: Heart Disease Prediction

## การใช้ Stacking Ensemble
วิธีการ ensemble ที่ผสมประสิทธิภาพของ LightGBM, XGBoost, CatBoost ผ่าน Meta-Learner
เพื่อให้ได้การทำนายที่แม่นยำที่สุด

In [ ]:
!pip install -q kaggle lightgbm xgboost catboost

In [ ]:
import os
from google.colab import files

# อัปโหลด kaggle.json
uploaded = files.upload()
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# ดาวน์โหลดข้อมูลจาก Kaggle
!kaggle competitions download -c super-ai-engineer-ss-6-heart-disease-prediction
!unzip -qo super-ai-engineer-ss-6-heart-disease-prediction.zip -d data

## Imports and Constants

In [ ]:
from __future__ import annotations
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
TARGET_COL = "History of HeartDisease or Attack"
BINARY_COLS = [
    "High Blood Pressure", "Told High Cholesterol", "Cholesterol Checked",
    "Smoked 100+ Cigarettes", "Diagnosed Stroke", "Diagnosed Diabetes",
    "Leisure Physical Activity", "Heavy Alcohol Consumption",
    "Health Care Coverage", "Doctor Visit Cost Barrier",
    "Difficulty Walking", "Vegetable or Fruit Intake (1+ per Day)",
]
N_FOLDS = 5
SEEDS = [42, 123, 2024, 7, 999]

## Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_ids = test["ID"].copy()

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Target:\n{train[TARGET_COL].value_counts()}")

## Feature Encoding

In [ ]:
def encode_features(df):
    """Encode: binary Yes/No -> 1/0, ordinal for health/edu/income"""
    df = df.copy()
    for col in BINARY_COLS:
        if col in df.columns:
            df[col] = df[col].map({"Yes": 1, "No": 0})
    if "Sex" in df.columns:
        df["Sex"] = df["Sex"].map({"Male": 1, "Female": 0})
    health_map = {"Very Poor": 0, "Poor": 1, "Fair": 2, "Good": 3, "Excellent": 4}
    if "General Health" in df.columns:
        df["General Health"] = df["General Health"].map(health_map)
    edu_map = {"Never attended school": 0, "Elementary": 1, "Some high school": 2,
               "High school graduate": 3, "Some college or technical school": 4, "College graduate": 5}
    if "Education Level" in df.columns:
        df["Education Level"] = df["Education Level"].map(edu_map)
    income_map = {"Less than $10,000": 0,
                  "$10,000 to less than $15,000": 1, "($10,000 to less than $15,000": 1,
                  "$15,000 to less than $20,000": 2, "($15,000 to less than $20,000": 2,
                  "$20,000 to less than $25,000": 3, "($20,000 to less than $25,000": 3,
                  "$25,000 to less than $35,000": 4, "($25,000 to less than $35,000": 4,
                  "$35,000 to less than $50,000": 5, "($35,000 to less than $50,000": 5,
                  "$50,000 to less than $75,000": 6, "($50,000 to less than $75,000": 6,
                  "$75,000 or more": 7}
    if "Income Level" in df.columns:
        df["Income Level"] = df["Income Level"].map(income_map)
    return df

# Prepare target
y_raw = train[TARGET_COL].map({"Yes": 1, "No": 0})
valid_mask = ~y_raw.isna()
train = train.loc[valid_mask].reset_index(drop=True)
y = y_raw.loc[valid_mask].astype(int).to_numpy()

train_enc = encode_features(train)
test_enc = encode_features(test)
print(f"Valid rows: {len(y)}, Positive rate: {y.mean():.4f}")

## Feature Engineering

In [ ]:
def safe_col(df, col, default):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(default)
    return pd.Series(default, index=df.index, dtype=np.float32)

def create_features(df):
    df = df.copy()
    age = safe_col(df, "Age", 55)
    bmi = safe_col(df, "Body Mass Index", 28)
    hbp = safe_col(df, "High Blood Pressure", 0)
    chol = safe_col(df, "Told High Cholesterol", 0)
    chol_checked = safe_col(df, "Cholesterol Checked", 0)
    smoke = safe_col(df, "Smoked 100+ Cigarettes", 0)
    stroke = safe_col(df, "Diagnosed Stroke", 0)
    diabetes = safe_col(df, "Diagnosed Diabetes", 0)
    walk = safe_col(df, "Difficulty Walking", 0)
    physical = safe_col(df, "Leisure Physical Activity", 0)
    alcohol = safe_col(df, "Heavy Alcohol Consumption", 0)
    fruit_veg = safe_col(df, "Vegetable or Fruit Intake (1+ per Day)", 0)
    general_health = safe_col(df, "General Health", 2)
    sex = safe_col(df, "Sex", 0)
    health_coverage = safe_col(df, "Health Care Coverage", 0)
    cost_barrier = safe_col(df, "Doctor Visit Cost Barrier", 0)
    education = safe_col(df, "Education Level", 3)
    income = safe_col(df, "Income Level", 4)

    df["metabolic_score"] = hbp + chol + diabetes + (bmi > 30).astype(int)
    df["cv_risk_bundle"] = hbp + chol + stroke + diabetes + walk
    df["total_risk_factors"] = hbp + chol + smoke + stroke + diabetes + walk + alcohol
    df["lifestyle_protection"] = physical + fruit_veg - alcohol - smoke

    df["age_bmi"] = age * bmi
    df["age_health"] = age * (4 - general_health)
    df["age_risk"] = age * (hbp + chol + diabetes)
    df["age_squared"] = age ** 2

    df["bmi_sedentary"] = bmi * (1 - physical)
    df["bmi_squared"] = bmi ** 2

    df["smoking_vascular"] = smoke * (hbp + stroke + diabetes)

    df["diabetes_cholesterol"] = diabetes * chol
    df["hbp_chol"] = hbp * chol
    df["stroke_hbp"] = stroke * hbp
    df["diabetes_hbp"] = diabetes * hbp
    df["walk_age"] = walk * age

    df["male_older"] = sex * (age > 55).astype(int)
    df["male_hbp"] = sex * hbp

    df["bmi_underweight"] = (bmi < 18.5).astype(int)
    df["bmi_normal"] = ((bmi >= 18.5) & (bmi < 25)).astype(int)
    df["bmi_overweight"] = ((bmi >= 25) & (bmi < 30)).astype(int)
    df["bmi_obese"] = ((bmi >= 30) & (bmi < 35)).astype(int)
    df["bmi_severely_obese"] = (bmi >= 35).astype(int)

    df["age_young"] = (age < 40).astype(int)
    df["age_middle"] = ((age >= 40) & (age < 55)).astype(int)
    df["age_senior"] = ((age >= 55) & (age < 65)).astype(int)
    df["age_elderly"] = (age >= 65).astype(int)

    df["healthcare_access"] = health_coverage + (1 - cost_barrier) + chol_checked

    df["low_ses_risk"] = ((income <= 2).astype(int) + (education <= 2).astype(int)) * (4 - general_health)

    framingham = age * (1 + hbp) * (1 + chol) * (1 + smoke) * (1 + diabetes)
    df["log_framingham"] = np.log1p(framingham)
    return df

train_feat = create_features(train_enc)
test_feat = create_features(test_enc)

drop_cols = {"ID", TARGET_COL}
feature_cols = [c for c in train_feat.columns if c not in drop_cols]
X_train = train_feat[feature_cols].to_numpy(dtype=np.float32)
X_test = test_feat.reindex(columns=feature_cols).to_numpy(dtype=np.float32)
X_train = np.nan_to_num(X_train, nan=-999.0, posinf=999999.0, neginf=-999999.0)
X_test = np.nan_to_num(X_test, nan=-999.0, posinf=999999.0, neginf=-999999.0)

pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
print(f"Features: {X_train.shape[1]}, Positive weight: {pos_weight:.2f}")

## Threshold Search Functions

In [ ]:
def find_best_threshold(y_true, probs, start=0.01, stop=0.90, step=0.005):
    best_f2, best_t = -1.0, start
    for t in np.arange(start, stop + 1e-12, step):
        preds = (probs > t).astype(int)
        f2 = fbeta_score(y_true, preds, beta=2)
        if f2 > best_f2:
            best_f2 = f2
            best_t = t
    return best_f2, best_t

def refine_threshold(y_true, probs, coarse_t):
    return find_best_threshold(y_true, probs,
        start=max(0.001, coarse_t - 0.03),
        stop=min(0.99, coarse_t + 0.03), step=0.001)

## Train Base Models (Multi-Seed K-Fold)

In [ ]:
base_models = {
    "lgb": lambda seed: lgb.LGBMClassifier(
        n_estimators=1500, learning_rate=0.02, max_depth=-1, num_leaves=63,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.3, reg_lambda=1.0,
        min_child_samples=30, is_unbalance=True, random_state=seed, n_jobs=-1, verbosity=-1),
    "xgb": lambda seed: xgb.XGBClassifier(
        n_estimators=1200, learning_rate=0.02, max_depth=6, subsample=0.8,
        colsample_bytree=0.7, min_child_weight=3, reg_alpha=0.3, reg_lambda=1.5,
        scale_pos_weight=pos_weight, random_state=seed, n_jobs=-1, verbosity=0,
        objective="binary:logistic", eval_metric="logloss", tree_method="hist"),
    "cb": lambda seed: CatBoostClassifier(
        iterations=1500, depth=6, learning_rate=0.03, l2_leaf_reg=5.0,
        loss_function="Logloss", eval_metric="Logloss", auto_class_weights="Balanced",
        random_seed=seed, verbose=False, task_type="CPU", thread_count=-1),
}

model_names = list(base_models.keys())
n_models = len(model_names)
oof_meta = np.zeros((len(X_train), n_models), dtype=np.float64)
test_meta = np.zeros((len(X_test), n_models), dtype=np.float64)

for m_idx, (name, model_fn) in enumerate(base_models.items()):
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")
    model_oof = np.zeros(len(X_train), dtype=np.float64)
    model_test = np.zeros(len(X_test), dtype=np.float64)

    for seed in SEEDS:
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y), start=1):
            model = model_fn(seed)
            if name == "lgb":
                model.fit(X_train[tr_idx], y[tr_idx],
                    eval_set=[(X_train[val_idx], y[val_idx])],
                    callbacks=[lgb.early_stopping(100, verbose=False)])
            elif name == "cb":
                model.fit(X_train[tr_idx], y[tr_idx],
                    eval_set=(X_train[val_idx], y[val_idx]), use_best_model=True)
            elif name == "xgb":
                model.fit(X_train[tr_idx], y[tr_idx],
                    eval_set=[(X_train[val_idx], y[val_idx])], verbose=False)
            model_oof[val_idx] += model.predict_proba(X_train[val_idx])[:, 1] / len(SEEDS)
            model_test += model.predict_proba(X_test)[:, 1] / (N_FOLDS * len(SEEDS))
        print(f"  seed {seed}: done")

    oof_meta[:, m_idx] = model_oof
    test_meta[:, m_idx] = model_test
    f2, t = find_best_threshold(y, model_oof)
    print(f"  {name} OOF F2: {f2:.5f} @ threshold {t:.3f}")

## Meta-Learner (Stacking)

In [ ]:
meta_oof = np.zeros(len(X_train), dtype=np.float64)
meta_test = np.zeros(len(X_test), dtype=np.float64)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_idx, val_idx) in enumerate(skf.split(oof_meta, y), start=1):
    meta_model = LogisticRegression(class_weight="balanced", C=1.0, random_state=42, max_iter=3000)
    meta_model.fit(oof_meta[tr_idx], y[tr_idx])
    meta_oof[val_idx] = meta_model.predict_proba(oof_meta[val_idx])[:, 1]
    meta_test += meta_model.predict_proba(test_meta)[:, 1] / 5
    print(f"  meta fold {fold}: done")

meta_f2, meta_t = find_best_threshold(y, meta_oof)
print(f"Meta OOF F2: {meta_f2:.5f} @ threshold {meta_t:.3f}")

## Strategy Search

In [ ]:
candidates = {}
candidates["meta"] = (meta_oof, meta_test)
for idx, name in enumerate(model_names):
    candidates[f"{name}_solo"] = (oof_meta[:, idx], test_meta[:, idx])
candidates["avg_base"] = (oof_meta.mean(axis=1), test_meta.mean(axis=1))

for alpha in np.arange(0.1, 1.0, 0.05):
    for idx, name in enumerate(model_names):
        blend_oof = alpha * meta_oof + (1 - alpha) * oof_meta[:, idx]
        blend_test = alpha * meta_test + (1 - alpha) * test_meta[:, idx]
        candidates[f"blend_meta_{name}_{alpha:.2f}"] = (blend_oof, blend_test)

for w_lgb in np.arange(0.2, 0.8, 0.1):
    for w_xgb in np.arange(0.1, 0.8 - w_lgb + 0.01, 0.1):
        w_cb = 1.0 - w_lgb - w_xgb
        if w_cb < 0.05:
            continue
        blend_oof = w_lgb * oof_meta[:, 0] + w_xgb * oof_meta[:, 1] + w_cb * oof_meta[:, 2]
        blend_test = w_lgb * test_meta[:, 0] + w_xgb * test_meta[:, 1] + w_cb * test_meta[:, 2]
        candidates[f"weighted_{w_lgb:.1f}_{w_xgb:.1f}_{w_cb:.1f}"] = (blend_oof, blend_test)

best_f2, best_thresh, best_name, best_test_probs = -1.0, 0.5, "", None
for name, (oof, test_probs_c) in candidates.items():
    f2, t = find_best_threshold(y, oof)
    if f2 > best_f2:
        best_f2, best_thresh, best_name, best_test_probs = f2, t, name, test_probs_c

best_oof = candidates[best_name][0]
best_f2, best_thresh = refine_threshold(y, best_oof, best_thresh)

print(f"\nBest strategy: {best_name}")
print(f"Best OOF F2: {best_f2:.5f}")
print(f"Best threshold: {best_thresh:.4f}")

## Generate Submission

In [ ]:
final_preds = (best_test_probs > best_thresh).astype(int)
pred_labels = pd.Series(final_preds).map({1: "Yes", 0: "No"})

submission = pd.DataFrame({"ID": test_ids, TARGET_COL: pred_labels})
submission.to_csv("submission.csv", index=False)

yes_count = int(final_preds.sum())
total = len(final_preds)
print(f"Rows: {total}")
print(f"Yes: {yes_count} ({100*yes_count/total:.1f}%), No: {total-yes_count}")

print(f"\nAll model OOF F2 scores:")
for idx, name in enumerate(model_names):
    f2, t = find_best_threshold(y, oof_meta[:, idx])
    print(f"  {name}: F2={f2:.5f} @ t={t:.4f}")

from google.colab import files
files.download("submission.csv")